In [8]:
# marginal shapley -- MMNIST / PolyMNIST version with TRANSLATED variant
# (3 modalities, matching MMVM paper Appendix B.4.1: "We only use the
#  first three modalities in this work.")
#
# CHANGES IN THIS REVISION (matched-data filtered + sweep start point)
# --------------------------------------------------------------------
# Two things relative to the previous matched-data revision:
#
# 1. DATA ROUTING (unchanged from previous revision):
#    All three families (benchmarks, joint marginal, joint shapley)
#    train AND test on the SAME filtered slice (rows where every
#    modality is present). No 5x training-data advantage for joint
#    methods. The data has no sentinels, so missing_value=None.
#
# 2. PREPROCESSING (new -- starting point for a sweep):
#    The previous preprocessing (9-row bands, asymmetric noise
#    [gaussian 0.20 / clean / dropout 0.25]) left modality 2 nearly
#    sufficient on its own (~0.33 acc), which is why late fusion
#    won: it picked up modality 2's signal essentially for free.
#    The new preprocessing aims for the regime where NO single
#    modality is sufficient but the COMBINATION is, so the joint
#    loss has cross-modal structure to exploit:
#
#      a. NARROWER, MORE SEPARATED BANDS:
#           m0: rows  0-5   (6 rows, top)
#           m1: rows 11-16  (6 rows, middle)
#           m2: rows 22-27  (6 rows, bottom)
#         Big gaps between bands (5 rows of black between m0/m1, 5
#         rows between m1/m2). Each modality sees ~21% of rows
#         instead of ~32%. Critically, the middle band (m1) is now
#         narrower so it can't carry the prediction alone the way
#         it did before.
#
#      b. SYMMETRIC GAUSSIAN NOISE:
#           all three modalities: gaussian, std=0.30
#         Equal noise on all three prevents any modality from being
#         the obvious "best single" winner. Asymmetric noise was
#         what let late-fusion-best-single match the ensemble in
#         the previous run.
#
#      c. LONGER TRAINING + STRONGER RHO SWEEP:
#           epochs:    30 -> 60
#           rho_list: [0.1, 0.5, 1.0, 2.0] -> [0.5, 1.0, 2.0, 5.0]
#         The joint trainer needs enough capacity to actually learn
#         cross-modal interactions; on ~1000 train rows with weak
#         per-modality models, 30 epochs may not be enough.
#
#    Target zone: each modality individually should land in
#    ~0.15-0.22 accuracy (above chance, below useful-on-its-own).
#    If you see m_i alone > 0.25, NOISE_LEVEL is too low. If you
#    see m_i alone < 0.13, it's too high.
#
# What to look at in the output (per repetition):
#    benchmarks/modality_{1,2,3}            -> per-modality accuracy alone
#    benchmarks/late_fusion_simple_average  -> the headline late-fusion baseline
#    benchmarks/late_fusion_best_single     -> just picks the best modality
#    joint/marginal/simple_average          -> joint baseline
#    joint/shapley/simple_average           -> joint with subset weighting
#
# Disk layout assumed:
#   MMNIST_ROOT/
#     train/m{0..4}/{within_class_id}.{digit}.png   (28x28 RGB, FLAT)
#     test /m{0..4}/{within_class_id}.{digit}.png

import os
import copy
import random
from typing import List, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

from meta_fusion.utils import *
from meta_fusion.models import *
from meta_fusion.methods import *
from meta_fusion.methodsextra_new import *
from meta_fusion.benchmarks import *

# ============================================================
# USER SETTINGS
# ============================================================

MMNIST_ROOT = "./polymnist/MMNIST"
DATA_DIR = "./polymnist_experiment_data"
os.makedirs(DATA_DIR, exist_ok=True)

MISSING_VALUE = -999.0

RANDOM_STATE = 42
USE_GPU = torch.cuda.is_available()
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20
BATCH_SIZE = 256

NUM_REPETITIONS = 20
REPETITION_SEEDS = [RANDOM_STATE + i for i in range(NUM_REPETITIONS)]

NUM_MODALITIES = 4
NUM_CLASSES = 10
IMAGE_HW = 28
IMAGE_CHANNELS = 3

N_SAMPLES_TOTAL = 8000

TRANSLATION_MODE = "per_modality_independent"
TRANSLATION_MAX_SHIFT_PIXELS = 3
TRANSLATION_SEED_SALT = "polymnist_translate_v1"

# ============================================================
# COMPLEMENTARITY PREPROCESSING -- SWEEP STARTING POINT
# ============================================================
# These are the values to start the sweep from. See the iteration
# guide at the bottom of this comment block for how to adjust.
APPLY_COMPLEMENTARITY = True

# NARROWER BANDS, BIGGER GAPS than before.
# Previous: (0,8) (10,17) (19,27) -> 26/28 rows covered, 1-row gaps.
# New:      (0,5) (11,16) (22,27) -> 18/28 rows covered, 5-row gaps.
# The middle band (m1) is now narrow enough that it can't carry the
# prediction on its own.
COMPLEMENTARITY_REGIONS = [
    (6, 15),    # m0: was (4,13), shifted down 2 rows
    (9, 18),    # m1: middle (unchanged)
    (14, 23),   # m2: lower-middle (unchanged)
        (6, 15),    # m0: was (4,13), shifted down 2 rows
]

# SYMMETRIC GAUSSIAN NOISE on all three modalities.
# Previous: ["gaussian", "none", "dropout"] @ [0.20, 0.0, 0.25]
#   -> m1 was clean and dominated (acc ~0.33 alone)
# New: all three gaussian @ 0.30
#   -> no modality has an unfair leg up; the joint loss has to do
#      real work to recover the digit from the union of bands.
APPLY_MODALITY_NOISE = True
MODALITY_NOISE_TYPES = ["gaussian", "gaussian", "gaussian", "gaussian"]
MODALITY_NOISE_LEVELS = [0.25, 0.25, 0.25, 0.25]
MODALITY_NOISE_SEED = 1234

APPLY_CHANNEL_SPLIT = False
CHANNEL_PER_MODALITY = [0, 1, 2]

# ============================================================
# Grouped missingness pattern (preserved -- still injected into the
# full pool, then we filter to fully-observed rows for this revision)
# ============================================================
GROUP_SHARE_ALL = 0.20
GROUP_SHARE_M2_MISS = 0.30
GROUP_SHARE_M1_MISS = 0.30
GROUP_SHARE_M0_MISS = 0.20

HIDDEN_DIMS_PER_MODALITY = [128, 64]
HIDDEN_DIMS_EARLY_FUSION = [256, 128, 64]

BASE_CONFIG = {
    "task_type": "classification",
    "output_dim": NUM_CLASSES,
    "use_gpu": USE_GPU,
    # STRONGER rho sweep. Previous: [0.1, 0.5, 1.0, 2.0]. With weak
    # per-modality models and small training sets, the joint loss
    # benefits from larger Shapley/marginal regularization.
    "rho_list":     [0.5, 1.0, 2.0, 5.0],
    "rho_list_ncl": [0.5, 1.0, 2.0, 5.0],
    # LONGER training. Previous: 30. The joint trainer needs more
    # passes to actually learn cross-modal structure when each
    # modality is individually weak.
    "epochs": 60,
    "init_lr": 1e-3,
    "weight_decay": 1e-4,
    "gamma": 1.0,
    "divergence_weight_type": "uniform",
    "burn_in_epochs": 0,
    "optimal_k": 2,
    "divergence_weight_scale": 1.0,
    "ensemble_methods": [
        "simple_average",
        "weighted_average",
        "majority_voting",
        "weighted_voting",
        "greedy_ensemble",
    ],
    "epochs_meta_learner": 20,
    "progress": False,
    "random_state": RANDOM_STATE,
    "verbose": True,
    "ckpt_dir": os.path.join(DATA_DIR, f"checkpoints_seed_{RANDOM_STATE}"),
}

LATE_FUSION_ENSEMBLE_METHODS = [
    "simple_average",
    "weighted_average",
    "majority_voting",
    "weighted_voting",
    "best_single",
    "greedy_ensemble",
]

# ============================================================
# BASIC MODELS
# ============================================================


class BenchmarkSingleModalityModel(nn.Module):
    def __init__(self, modality_idx: int, input_dim: int, hidden_dims: List[int], output_dim: int):
        super().__init__()
        self.modality_idx = modality_idx
        self.model = MLP_Net(input_dim, hidden_dims, output_dim)

    def forward(self, modalities):
        x = modalities[self.modality_idx]
        return self.model(x)


class BenchmarkEarlyFusionModel(nn.Module):
    def __init__(self, input_dims: List[int], hidden_dims: List[int], output_dim: int):
        super().__init__()
        self.model = MLP_Net(sum(input_dims), hidden_dims, output_dim)

    def forward(self, modalities):
        x = torch.cat(modalities, dim=1)
        return self.model(x)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_config_for_seed(split_seed: int):
    config = copy.deepcopy(BASE_CONFIG)
    config["random_state"] = split_seed
    config["ckpt_dir"] = os.path.join(DATA_DIR, f"checkpoints_split_seed_{split_seed}")
    os.makedirs(config["ckpt_dir"], exist_ok=True)
    return config


# ============================================================
# DATASETS
# ============================================================


class PolyMNISTTensorDataset(Dataset):
    def __init__(self, arrays: List[np.ndarray], y: np.ndarray):
        self.arrays = [torch.tensor(a, dtype=torch.float32) for a in arrays]
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (*[a[idx] for a in self.arrays], self.y[idx])


AlzheimerTensorDataset = PolyMNISTTensorDataset


class SingleModalityTensorDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# ============================================================
# COMPLEMENTARITY PREPROCESSING
# ============================================================


def apply_complementarity_to_arrays(
    arrays: List[np.ndarray],
    regions: List[Tuple[int, int]] = None,
    channels: List[int] = None,
    image_hw: int = IMAGE_HW,
    image_channels: int = IMAGE_CHANNELS,
) -> List[np.ndarray]:
    if regions is None:
        regions = COMPLEMENTARITY_REGIONS
    if channels is None and APPLY_CHANNEL_SPLIT:
        channels = CHANNEL_PER_MODALITY
    if len(regions) != len(arrays):
        raise ValueError(
            f"Need one region per modality, got {len(regions)} regions for {len(arrays)} modalities."
        )
    if channels is not None and len(channels) != len(arrays):
        raise ValueError(
            f"Need one channel per modality, got {len(channels)} channels for {len(arrays)} modalities."
        )

    out = []
    print("=" * 80)
    print("Applying complementarity preprocessing (spatial bands"
          + (" + channel split" if channels is not None else "") + ")")
    for m_idx, (arr, (r0, r1)) in enumerate(zip(arrays, regions)):
        n = arr.shape[0]
        feature_dim = image_hw * image_hw * image_channels
        if arr.shape[1] != feature_dim:
            raise ValueError(
                f"Modality {m_idx} has feature_dim {arr.shape[1]}, expected {feature_dim}."
            )
        img = arr.reshape(n, image_hw, image_hw, image_channels).copy()

        spatial_mask = np.zeros((image_hw, image_hw), dtype=np.float32)
        spatial_mask[r0:r1 + 1, :] = 1.0
        img = img * spatial_mask[None, :, :, None]

        ch_str = "all_RGB"
        if channels is not None:
            ch = int(channels[m_idx])
            if ch < 0 or ch >= image_channels:
                raise ValueError(f"Channel index {ch} for modality {m_idx} out of range.")
            ch_mask = np.zeros((image_channels,), dtype=np.float32)
            ch_mask[ch] = 1.0
            img = img * ch_mask[None, None, None, :]
            ch_str = ["R", "G", "B"][ch]

        out.append(img.reshape(n, -1).astype(np.float32))
        kept_rows = int((r1 - r0 + 1))
        pct_rows = 100.0 * kept_rows / image_hw
        pct_total = pct_rows * (1.0 / image_channels if channels is not None else 1.0)
        print(f"  m{m_idx}: rows {r0}..{r1} ({kept_rows}/{image_hw} = {pct_rows:.1f}%), "
              f"channel={ch_str}  ->  ~{pct_total:.1f}% of original pixel info")
    print("=" * 80)
    return out


def apply_modality_noise(
    arrays: List[np.ndarray],
    noise_types: List[str] = None,
    noise_levels: List[float] = None,
    seed: int = MODALITY_NOISE_SEED,
    image_hw: int = IMAGE_HW,
    image_channels: int = IMAGE_CHANNELS,
    regions: List[Tuple[int, int]] = None,
    channels: List[int] = None,
) -> List[np.ndarray]:
    if noise_types is None:
        noise_types = MODALITY_NOISE_TYPES
    if noise_levels is None:
        noise_levels = MODALITY_NOISE_LEVELS
    if regions is None:
        regions = COMPLEMENTARITY_REGIONS
    if channels is None and APPLY_CHANNEL_SPLIT:
        channels = CHANNEL_PER_MODALITY

    if len(noise_types) != len(arrays) or len(noise_levels) != len(arrays):
        raise ValueError("noise_types and noise_levels must each have one entry per modality.")

    out = []
    print("=" * 80)
    print("Applying per-modality noise (deterministic)")
    for m_idx, arr in enumerate(arrays):
        ntype = noise_types[m_idx]
        level = float(noise_levels[m_idx])
        n = arr.shape[0]
        img = arr.reshape(n, image_hw, image_hw, image_channels).copy()

        r0, r1 = regions[m_idx]
        spatial_mask = np.zeros((image_hw, image_hw), dtype=np.float32)
        spatial_mask[r0:r1 + 1, :] = 1.0
        survival = spatial_mask[None, :, :, None]
        if channels is not None:
            ch = int(channels[m_idx])
            ch_mask = np.zeros((image_channels,), dtype=np.float32)
            ch_mask[ch] = 1.0
            survival = survival * ch_mask[None, None, None, :]

        rng = np.random.RandomState(seed + m_idx * 7919)

        if ntype == "none" or level <= 0:
            print(f"  m{m_idx}: no noise (anchor)")
        elif ntype == "gaussian":
            noise = rng.normal(loc=0.0, scale=level, size=img.shape).astype(np.float32)
            img = img + noise * survival
            img = np.clip(img, 0.0, 1.0)
            print(f"  m{m_idx}: gaussian noise, std={level}")
        elif ntype == "dropout":
            keep = (rng.random_sample(size=img.shape).astype(np.float32) >= level).astype(np.float32)
            keep_eff = keep * survival + (1.0 - survival)
            img = img * keep_eff
            print(f"  m{m_idx}: pixel dropout, p={level}")
        else:
            raise ValueError(f"Unknown noise type for modality {m_idx}: {ntype!r}")

        out.append(img.reshape(n, -1).astype(np.float32))
    print("=" * 80)
    return out


# ============================================================
# MISSINGNESS HELPERS
# ============================================================


def get_modality_fully_missing_mask(a: np.ndarray, missing_value: float) -> np.ndarray:
    return np.all(a == missing_value, axis=1)


def get_no_missingness_mask(arrays: List[np.ndarray], missing_value: float) -> np.ndarray:
    keep = np.ones(arrays[0].shape[0], dtype=bool)
    for a in arrays:
        keep &= ~get_modality_fully_missing_mask(a, missing_value)
    return keep


def inject_grouped_missingness(
    arrays: List[np.ndarray],
    missing_value: float,
    random_state: int,
    share_all: float = GROUP_SHARE_ALL,
    share_m2_miss: float = GROUP_SHARE_M2_MISS,
    share_m1_miss: float = GROUP_SHARE_M1_MISS,
    share_m0_miss: float = GROUP_SHARE_M0_MISS,
) -> List[np.ndarray]:
    total = share_all + share_m2_miss + share_m1_miss + share_m0_miss
    if not np.isclose(total, 1.0, atol=1e-6):
        raise ValueError(f"Group shares must sum to 1.0, got {total}")
    if len(arrays) != 3:
        raise ValueError(
            "inject_grouped_missingness assumes exactly 3 modalities. "
            f"Got {len(arrays)}."
        )

    arrays = [a.copy() for a in arrays]
    n = arrays[0].shape[0]
    rng = np.random.RandomState(random_state)

    n_all = int(round(share_all * n))
    n_m2_miss = int(round(share_m2_miss * n))
    n_m1_miss = int(round(share_m1_miss * n))
    n_m0_miss = n - n_all - n_m2_miss - n_m1_miss
    if n_m0_miss < 0:
        n_m0_miss = 0
        n_m1_miss = n - n_all - n_m2_miss

    perm = rng.permutation(n)
    cuts = np.cumsum([n_all, n_m2_miss, n_m1_miss, n_m0_miss])
    idx_all = perm[:cuts[0]]
    idx_m2_miss = perm[cuts[0]:cuts[1]]
    idx_m1_miss = perm[cuts[1]:cuts[2]]
    idx_m0_miss = perm[cuts[2]:cuts[3]]

    arrays[2][idx_m2_miss, :] = missing_value
    arrays[1][idx_m1_miss, :] = missing_value
    arrays[0][idx_m0_miss, :] = missing_value

    print("=" * 80)
    print("Injected grouped missingness (4-way partition)")
    print(f"  All modalities present   : {len(idx_all):>6d} rows ({100*len(idx_all)/n:5.1f}%)")
    print(f"  m0,m1 present (m2 miss)  : {len(idx_m2_miss):>6d} rows ({100*len(idx_m2_miss)/n:5.1f}%)")
    print(f"  m0,m2 present (m1 miss)  : {len(idx_m1_miss):>6d} rows ({100*len(idx_m1_miss)/n:5.1f}%)")
    print(f"  m1,m2 present (m0 miss)  : {len(idx_m0_miss):>6d} rows ({100*len(idx_m0_miss)/n:5.1f}%)")
    print(f"  Total                    : {n:>6d} rows")
    print("=" * 80)
    return arrays


# ============================================================
# TRANSLATION HELPERS
# ============================================================


def _deterministic_offset(modality_idx: int, sample_key: str, max_shift: int) -> Tuple[int, int]:
    rng = random.Random((TRANSLATION_SEED_SALT, int(modality_idx), str(sample_key)))
    dx = rng.randint(-max_shift, max_shift)
    dy = rng.randint(-max_shift, max_shift)
    return dx, dy


def _shift_image_with_zero_pad(img: np.ndarray, dx: int, dy: int) -> np.ndarray:
    H, W = img.shape[:2]
    out = np.zeros_like(img)
    if abs(dx) >= W or abs(dy) >= H:
        return out

    src_y0 = max(0, -dy)
    src_y1 = min(H, H - dy)
    src_x0 = max(0, -dx)
    src_x1 = min(W, W - dx)

    dst_y0 = max(0, dy)
    dst_y1 = dst_y0 + (src_y1 - src_y0)
    dst_x0 = max(0, dx)
    dst_x1 = dst_x0 + (src_x1 - src_x0)

    out[dst_y0:dst_y1, dst_x0:dst_x1, :] = img[src_y0:src_y1, src_x0:src_x1, :]
    return out


def _translation_for(split_name: str, fname: str, modality_idx: int) -> Tuple[int, int]:
    if TRANSLATION_MODE == "off":
        return 0, 0
    if TRANSLATION_MODE == "per_modality_independent":
        sample_key = f"{split_name}/{fname}/m{modality_idx}"
        return _deterministic_offset(modality_idx, sample_key, TRANSLATION_MAX_SHIFT_PIXELS)
    if TRANSLATION_MODE == "shared_across_modalities":
        sample_key = f"{split_name}/{fname}"
        return _deterministic_offset(0, sample_key, TRANSLATION_MAX_SHIFT_PIXELS)
    raise ValueError(f"Unknown TRANSLATION_MODE: {TRANSLATION_MODE!r}")


# ============================================================
# REAL MMNIST LOADER
# ============================================================


def _list_aligned_filenames(split_dir: str, num_modalities: int):
    per_modality_sets = []
    for m in range(num_modalities):
        folder = os.path.join(split_dir, f"m{m}")
        if not os.path.isdir(folder):
            raise FileNotFoundError(f"Expected folder not found: {folder}")
        files = {f for f in os.listdir(folder) if f.endswith(".png")}
        per_modality_sets.append(files)

    common = set.intersection(*per_modality_sets)

    parsed = []
    for fname in common:
        stem = os.path.splitext(fname)[0]
        parts = stem.split(".")
        if len(parts) != 2:
            continue
        try:
            within_id = int(parts[0])
            digit = int(parts[1])
        except ValueError:
            continue
        if 0 <= digit <= 9:
            parsed.append((fname, within_id, digit))

    parsed.sort(key=lambda t: (t[2], t[1]))
    return [(fname, digit) for fname, _, digit in parsed]


def _read_png_as_rgb_array(path: str) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img, dtype=np.float32) / 255.0
    if arr.shape != (IMAGE_HW, IMAGE_HW, 3):
        img = img.resize((IMAGE_HW, IMAGE_HW), Image.BILINEAR)
        arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    return arr


def _load_split_aligned(split_dir: str, num_modalities: int, split_name: str):
    aligned = _list_aligned_filenames(split_dir, num_modalities)
    n = len(aligned)
    print(f"  found {n} aligned multimodal samples across {num_modalities} modalities")
    if n == 0:
        raise RuntimeError(f"No aligned filenames found under {split_dir}.")

    feature_dim = IMAGE_HW * IMAGE_HW * IMAGE_CHANNELS
    arrays = [np.empty((n, feature_dim), dtype=np.float32) for _ in range(num_modalities)]
    labels = np.empty(n, dtype=np.int64)
    per_class_count = {d: 0 for d in range(10)}

    translations_applied = 0
    nonzero_translations = 0

    for i, (fname, digit) in enumerate(aligned):
        labels[i] = digit
        per_class_count[digit] += 1
        for m in range(num_modalities):
            path = os.path.join(split_dir, f"m{m}", fname)
            img = _read_png_as_rgb_array(path)

            if TRANSLATION_MODE != "off":
                dx, dy = _translation_for(split_name, fname, m)
                if dx != 0 or dy != 0:
                    img = _shift_image_with_zero_pad(img, dx, dy)
                    nonzero_translations += 1
                translations_applied += 1

            arrays[m][i] = img.reshape(-1)

    print("  per-digit counts:", {d: per_class_count[d] for d in range(10)})
    if TRANSLATION_MODE != "off":
        total = n * num_modalities
        pct = 100.0 * nonzero_translations / max(total, 1)
        print(f"  translation: {translations_applied}/{total} images processed, "
              f"{nonzero_translations} ({pct:.1f}%) had a non-zero shift")
    return arrays, labels


_MMNIST_CACHE = {"loaded": False, "arrays": None, "labels": None}


def _load_mmnist_pool_once():
    if _MMNIST_CACHE["loaded"]:
        return _MMNIST_CACHE["arrays"], _MMNIST_CACHE["labels"]

    if not os.path.isdir(MMNIST_ROOT):
        raise FileNotFoundError(
            f"MMNIST_ROOT not found: {MMNIST_ROOT}\n"
            f"Set MMNIST_ROOT to the folder that contains train/ and test/."
        )
    train_dir = os.path.join(MMNIST_ROOT, "train")
    test_dir = os.path.join(MMNIST_ROOT, "test")

    print("=" * 80)
    print(f"Loading PolyMNIST from {MMNIST_ROOT} (one-time read, cached after this)")
    print(f"TRANSLATION_MODE = {TRANSLATION_MODE!r}, "
          f"max_shift = {TRANSLATION_MAX_SHIFT_PIXELS} pixels")
    print("=" * 80)
    print("Train split:")
    train_arrays, train_labels = _load_split_aligned(train_dir, NUM_MODALITIES, "train")
    print("Test split:")
    test_arrays, test_labels = _load_split_aligned(test_dir, NUM_MODALITIES, "test")

    arrays = [np.concatenate([tr, te], axis=0) for tr, te in zip(train_arrays, test_arrays)]
    labels = np.concatenate([train_labels, test_labels], axis=0).astype(int)

    _MMNIST_CACHE["loaded"] = True
    _MMNIST_CACHE["arrays"] = arrays
    _MMNIST_CACHE["labels"] = labels
    print(f"Cached pool: {len(labels)} aligned multimodal samples.")
    return arrays, labels


def load_polymnist_preprocessed():
    pool_arrays, pool_y = _load_mmnist_pool_once()

    rng = np.random.RandomState(RANDOM_STATE)
    if N_SAMPLES_TOTAL is not None and N_SAMPLES_TOTAL < len(pool_y):
        sub_idx = rng.choice(len(pool_y), size=N_SAMPLES_TOTAL, replace=False)
        sub_idx.sort()
        arrays = [a[sub_idx].copy() for a in pool_arrays]
        y = pool_y[sub_idx].copy()
        print(f"Subsampled to {len(y)} aligned multimodal samples.")
    else:
        arrays = [a.copy() for a in pool_arrays]
        y = pool_y.copy()
        print(f"Using all {len(y)} aligned multimodal samples.")

    if APPLY_COMPLEMENTARITY:
        arrays = apply_complementarity_to_arrays(
            arrays,
            regions=COMPLEMENTARITY_REGIONS,
            channels=(CHANNEL_PER_MODALITY if APPLY_CHANNEL_SPLIT else None),
        )
    else:
        print("=" * 80)
        print("APPLY_COMPLEMENTARITY = False : modalities kept as-is (redundant).")
        print("=" * 80)

    if APPLY_MODALITY_NOISE:
        arrays = apply_modality_noise(
            arrays,
            noise_types=MODALITY_NOISE_TYPES,
            noise_levels=MODALITY_NOISE_LEVELS,
            seed=MODALITY_NOISE_SEED,
            regions=COMPLEMENTARITY_REGIONS,
            channels=(CHANNEL_PER_MODALITY if APPLY_CHANNEL_SPLIT else None),
        )
    else:
        print("=" * 80)
        print("APPLY_MODALITY_NOISE = False : modalities kept noiseless.")
        print("=" * 80)

    subjects = np.arange(len(y))

    arrays = inject_grouped_missingness(
        arrays,
        missing_value=MISSING_VALUE,
        random_state=RANDOM_STATE,
    )
    return subjects, y, arrays


# ============================================================
# LOADER BUILDERS
# ============================================================


def make_loaders_from_arrays(train_arrays, val_arrays, test_arrays, y_train, y_val, y_test, batch_size):
    train_ds = PolyMNISTTensorDataset(train_arrays, y_train)
    val_ds = PolyMNISTTensorDataset(val_arrays, y_val)
    test_ds = PolyMNISTTensorDataset(test_arrays, y_test)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader, test_loader


def make_single_modality_loader(x, y, batch_size, shuffle, drop_last):
    ds = SingleModalityTensorDataset(x, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


# ============================================================
# METRICS
# ============================================================


def multiclass_sensitivity(y_true, y_pred, num_classes):
    vals = []
    for c in range(num_classes):
        yt = (y_true == c).astype(int)
        yp = (y_pred == c).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        vals.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
    return float(np.nanmean(vals))


def multiclass_specificity(y_true, y_pred, num_classes):
    vals = []
    for c in range(num_classes):
        yt = (y_true == c).astype(int)
        yp = (y_pred == c).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        vals.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
    return float(np.nanmean(vals))


def safe_multiclass_auc(y_true, y_prob):
    try:
        if len(np.unique(y_true)) < 2:
            return float("nan")
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
    except Exception:
        return float("nan")


def compute_classification_metrics(y_true, y_pred, y_prob, num_classes=NUM_CLASSES):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "auc_ovr": safe_multiclass_auc(y_true, y_prob),
        "sensitivity": multiclass_sensitivity(y_true, y_pred, num_classes),
        "specificity": multiclass_specificity(y_true, y_pred, num_classes),
    }


import meta_fusion.methodsextra_new as mf_extra
import meta_fusion.benchmarks as mf_benchmarks

mf_extra.multiclass_sensitivity = multiclass_sensitivity
mf_extra.multiclass_specificity = multiclass_specificity
mf_extra.safe_multiclass_auc = safe_multiclass_auc
mf_extra.compute_classification_metrics = compute_classification_metrics

mf_benchmarks.multiclass_sensitivity = multiclass_sensitivity
mf_benchmarks.multiclass_specificity = multiclass_specificity
mf_benchmarks.safe_multiclass_auc = safe_multiclass_auc
mf_benchmarks.compute_classification_metrics = compute_classification_metrics


# ============================================================
# SPLIT LOGIC -- MATCHED-DATA FILTERED VERSION
# ============================================================


def get_filtered_mask(arrays: List[np.ndarray], missing_value: float) -> np.ndarray:
    keep = np.ones(arrays[0].shape[0], dtype=bool)
    for a in arrays:
        keep &= ~(np.all(a == missing_value, axis=1))
    return keep


def build_subject_level_splits(subjects, y, test_size=0.2, val_size_within_train=0.2, random_state=42):
    subject_label_df = (
        pd.DataFrame({"subject": subjects, "label": y})
        .groupby("subject")["label"]
        .agg(lambda s: s.value_counts().index[0])
        .reset_index()
    )

    unique_subjects = subject_label_df["subject"].values
    unique_subject_labels = subject_label_df["label"].values

    subj_train, subj_test = train_test_split(
        unique_subjects,
        test_size=test_size,
        random_state=random_state,
        stratify=unique_subject_labels,
    )

    train_subject_labels = subject_label_df.set_index("subject").loc[subj_train]["label"].values
    subj_train, subj_val = train_test_split(
        subj_train,
        test_size=val_size_within_train,
        random_state=random_state,
        stratify=train_subject_labels,
    )

    train_idx = np.where(np.isin(subjects, subj_train))[0]
    val_idx = np.where(np.isin(subjects, subj_val))[0]
    test_idx = np.where(np.isin(subjects, subj_test))[0]
    return train_idx, val_idx, test_idx


def build_filtered_only_splits(subjects, y, arrays, missing_value, test_size,
                               val_size_within_train, random_state):
    """Filter to fully-observed rows and split them into train/val/test.
    All three method families share these splits.
    """
    filtered_mask = get_filtered_mask(arrays, missing_value)
    filtered_idx_global = np.where(filtered_mask)[0]
    print(f"Filtered (all modalities present) rows: {filtered_mask.sum()} / {len(filtered_mask)}")
    print(f"Discarded (>=1 modality missing) rows : {(~filtered_mask).sum()} / {len(filtered_mask)}")

    filtered_subjects = subjects[filtered_mask]
    filtered_y = y[filtered_mask]
    train_local, val_local, test_local = build_subject_level_splits(
        filtered_subjects, filtered_y,
        test_size=test_size, val_size_within_train=val_size_within_train,
        random_state=random_state,
    )
    train_idx = filtered_idx_global[train_local]
    val_idx = filtered_idx_global[val_local]
    test_idx = filtered_idx_global[test_local]

    print(f"  matched-data splits (shared by all families): "
          f"train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    return {
        "filtered_mask": filtered_mask,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
    }


def subset_by_indices(arrays, y, idx):
    return [a[idx] for a in arrays], y[idx]


def build_all_settings(split_seed: int):
    subjects, y, arrays = load_polymnist_preprocessed()

    split_info = build_filtered_only_splits(
        subjects=subjects, y=y, arrays=arrays, missing_value=MISSING_VALUE,
        test_size=TEST_SIZE, val_size_within_train=VAL_SIZE_WITHIN_TRAIN,
        random_state=split_seed,
    )

    train_arrays, y_train = subset_by_indices(arrays, y, split_info["train_idx"])
    val_arrays, y_val = subset_by_indices(arrays, y, split_info["val_idx"])
    test_arrays, y_test = subset_by_indices(arrays, y, split_info["test_idx"])

    return {
        "filtered": {
            "train_arrays": train_arrays,
            "val_arrays": val_arrays,
            "test_arrays": test_arrays,
            "y_train": y_train, "y_val": y_val, "y_test": y_test,
            "missing_value": None,
        },
    }


# ============================================================
# PATCHED TRAINERS
# ============================================================


class TrainerMetaFusionMetrics(Trainer_new):
    def test_classification(self, ensemble_methods, test_loader, best_val_task_losses):
        records = {m: {"y_true": [], "y_pred": [], "y_prob": []} for m in ensemble_methods}
        cohort_records = [{"y_true": [], "y_pred": [], "y_prob": []} for _ in range(self.model_num)]

        for i in range(self.model_num):
            self.models[i].eval()
        if "meta_learner" in ensemble_methods:
            self.meta_learner.eval()

        with torch.no_grad():
            for batch in test_loader:
                modalities, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    modalities = [mod.cuda() for mod in modalities]
                    target = target.cuda()

                outputs = []
                for i, model in enumerate(self.models):
                    output = model(modalities[i])
                    outputs.append(output)
                    prob = torch.softmax(output, dim=1).cpu().numpy()
                    pred = torch.argmax(output, dim=1).cpu().numpy()
                    cohort_records[i]["y_true"].append(target.cpu().numpy())
                    cohort_records[i]["y_pred"].append(pred)
                    cohort_records[i]["y_prob"].append(prob)

                outputs_stack = torch.stack(outputs)

                for method in ensemble_methods:
                    if method == "simple_average":
                        final_output = torch.mean(outputs_stack, dim=0)
                    elif method == "weighted_average":
                        weights = get_weights_by_task_loss(best_val_task_losses).to(outputs_stack.device)
                        final_output = torch.sum(weights.unsqueeze(1).unsqueeze(2) * outputs_stack, dim=0)
                    elif method == "majority_voting":
                        final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                        final_output = F.one_hot(final_pred, num_classes=self.num_classes).float()
                    elif method == "weighted_voting":
                        weights = get_weights_by_task_loss(best_val_task_losses).to(outputs_stack.device)
                        top_preds = torch.argmax(outputs_stack, dim=2)
                        num_classes = outputs_stack.shape[2]
                        batch_size = outputs_stack.shape[1]
                        weighted_votes = torch.zeros((batch_size, num_classes), device=outputs_stack.device)
                        for k, model_preds in enumerate(top_preds):
                            weighted_votes.scatter_add_(
                                1, model_preds.unsqueeze(1),
                                torch.full((batch_size, 1), float(weights[k]), device=outputs_stack.device),
                            )
                        final_output = weighted_votes
                    elif method == "meta_learner":
                        outputs_concat = torch.cat(outputs, dim=1)
                        final_output = self.meta_learner(outputs_concat)
                    elif method == "best_single":
                        best_model = best_val_task_losses.index(min(best_val_task_losses))
                        final_output = outputs_stack[best_model]
                    elif method == "greedy_ensemble":
                        weights = get_weights_by_task_loss(best_val_task_losses)[self.ens_idxs].to(outputs_stack.device)
                        weights = weights / torch.sum(weights)
                        final_output = torch.sum(
                            weights.unsqueeze(1).unsqueeze(2) * outputs_stack[self.ens_idxs], dim=0
                        )
                    else:
                        raise ValueError(method)

                    prob = torch.softmax(final_output, dim=1).cpu().numpy()
                    pred = torch.argmax(final_output, dim=1).cpu().numpy()
                    records[method]["y_true"].append(target.cpu().numpy())
                    records[method]["y_pred"].append(pred)
                    records[method]["y_prob"].append(prob)

        results = {}
        for method, rec in records.items():
            y_true = np.concatenate(rec["y_true"])
            y_pred = np.concatenate(rec["y_pred"])
            y_prob = np.concatenate(rec["y_prob"])
            results[method] = compute_classification_metrics(y_true, y_pred, y_prob, num_classes=self.num_classes)

        results["cohort"] = []
        for rec in cohort_records:
            y_true = np.concatenate(rec["y_true"])
            y_pred = np.concatenate(rec["y_pred"])
            y_prob = np.concatenate(rec["y_prob"])
            results["cohort"].append(
                compute_classification_metrics(y_true, y_pred, y_prob, num_classes=self.num_classes)
            )
        return results


class TrainerJointPolyMNIST(Trainer_Joint_new):
    def __init__(self, config, models, data_loaders):
        super().__init__(config, models, data_loaders)
        self.loss_mse = self.loss_task

    def test(self, test_loader, missing_value=None):
        self.missing_value = missing_value

        if "meta_learner" in self.ensemble_methods:
            self.meta_learner = self.initialize_meta_learner()
            self.meta_learner = self.meta_learner.to(self.device)
            self.meta_learner_optimizer = torch.optim.Adam(self.meta_learner.parameters(), lr=0.1, weight_decay=0)
            self.load_meta_learner()

        if self.task_type == "classification":
            best_val_task_losses = self.validate(missing_value=getattr(self, "missing_value", None))
            best_val_task_losses = [best_val_task_losses[i].avg for i in range(self.model_num)]
            return self.test_classification_metrics(
                self.ensemble_methods + ["best_single"],
                test_loader,
                best_val_task_losses,
            )
        else:
            return super().test(test_loader, missing_value=missing_value)

    def test_classification_metrics(self, ensemble_methods, test_loader, best_val_task_losses):
        records = {m: {"y_true": [], "y_pred": [], "y_prob": []} for m in ensemble_methods}
        cohort_records = [{"y_true": [], "y_pred": [], "y_prob": []} for _ in range(self.model_num)]

        for i in range(self.model_num):
            self.models[i].eval()
        if "meta_learner" in ensemble_methods:
            self.meta_learner.eval()

        with torch.no_grad():
            for batch in test_loader:
                modalities, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    modalities = [mod.cuda() for mod in modalities]
                    target = target.cuda()

                missing_value = getattr(self, "missing_value", None)
                weights_all = get_weights_by_task_loss(best_val_task_losses)

                if missing_value is None:
                    outputs = []
                    for i, model in enumerate(self.models):
                        output = model(modalities[i])
                        outputs.append(output)
                        prob = torch.softmax(output, dim=1).cpu().numpy()
                        pred = torch.argmax(output, dim=1).cpu().numpy()
                        cohort_records[i]["y_true"].append(target.cpu().numpy())
                        cohort_records[i]["y_pred"].append(pred)
                        cohort_records[i]["y_prob"].append(prob)

                    outputs_stack = torch.stack(outputs)

                    for method in ensemble_methods:
                        if method == "simple_average":
                            final_output = torch.mean(outputs_stack, dim=0)
                        elif method == "weighted_average":
                            final_output = torch.sum(
                                weights_all.to(outputs_stack.device).unsqueeze(1).unsqueeze(2) * outputs_stack,
                                dim=0,
                            )
                        elif method == "majority_voting":
                            final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                            final_output = F.one_hot(final_pred, num_classes=self.num_classes).float()
                        elif method == "weighted_voting":
                            top_preds = torch.argmax(outputs_stack, dim=2)
                            num_classes = outputs_stack.shape[2]
                            batch_size = outputs_stack.shape[1]
                            weighted_votes = torch.zeros((batch_size, num_classes), device=outputs_stack.device)
                            for k, model_preds in enumerate(top_preds):
                                weighted_votes.scatter_add_(
                                    1, model_preds.unsqueeze(1),
                                    torch.full((batch_size, 1), float(weights_all[k]), device=outputs_stack.device),
                                )
                            final_output = weighted_votes
                        elif method == "meta_learner":
                            outputs_concat = torch.cat(outputs, dim=1)
                            final_output = self.meta_learner(outputs_concat)
                        elif method == "best_single":
                            best_model = best_val_task_losses.index(min(best_val_task_losses))
                            final_output = outputs_stack[best_model]
                        elif method == "greedy_ensemble":
                            weights = get_weights_by_task_loss(best_val_task_losses)[self.ens_idxs]
                            weights = weights / torch.sum(weights)
                            weights = weights.unsqueeze(1).unsqueeze(2).to(outputs_stack.device)
                            final_output = torch.sum(weights * outputs_stack[self.ens_idxs], dim=0)
                        else:
                            raise ValueError(method)

                        prob = torch.softmax(final_output, dim=1).cpu().numpy()
                        pred = torch.argmax(final_output, dim=1).cpu().numpy()
                        records[method]["y_true"].append(target.cpu().numpy())
                        records[method]["y_pred"].append(pred)
                        records[method]["y_prob"].append(prob)
                else:
                    patterns = self._group_indices_by_availability(modalities, missing_value)

                    for pattern, idx in patterns.items():
                        present_models = [k for k, ok in enumerate(pattern) if ok]
                        if len(present_models) == 0:
                            continue

                        mods_g = [modalities[k][idx] for k in range(self.model_num)]
                        target_g = target[idx]
                        outputs_present = []

                        for mi in present_models:
                            out = self.models[mi](mods_g[mi])
                            outputs_present.append(out)
                            prob = torch.softmax(out, dim=1).cpu().numpy()
                            pred = torch.argmax(out, dim=1).cpu().numpy()
                            cohort_records[mi]["y_true"].append(target_g.cpu().numpy())
                            cohort_records[mi]["y_pred"].append(pred)
                            cohort_records[mi]["y_prob"].append(prob)

                        outputs_stack = torch.stack(outputs_present)

                        for method in ensemble_methods:
                            if method == "simple_average":
                                final_output = torch.mean(outputs_stack, dim=0)
                            elif method == "weighted_average":
                                w = weights_all[present_models]
                                w = w / torch.sum(w)
                                final_output = torch.sum(
                                    w.to(outputs_stack.device).unsqueeze(1).unsqueeze(2) * outputs_stack,
                                    dim=0,
                                )
                            elif method == "majority_voting":
                                final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                                final_output = F.one_hot(final_pred, num_classes=self.num_classes).float()
                            elif method == "weighted_voting":
                                w = weights_all[present_models]
                                w = w / torch.sum(w)
                                top_preds = torch.argmax(outputs_stack, dim=2)
                                num_classes = outputs_stack.shape[2]
                                batch_size = outputs_stack.shape[1]
                                weighted_votes = torch.zeros((batch_size, num_classes), device=outputs_stack.device)
                                for k, model_preds in enumerate(top_preds):
                                    weighted_votes.scatter_add_(
                                        1, model_preds.unsqueeze(1),
                                        torch.full((batch_size, 1), float(w[k]), device=outputs_stack.device),
                                    )
                                final_output = weighted_votes
                            elif method == "meta_learner":
                                full_outputs = []
                                for mi in range(self.model_num):
                                    if mi in present_models:
                                        j = present_models.index(mi)
                                        full_outputs.append(outputs_present[j])
                                    else:
                                        full_outputs.append(torch.zeros_like(outputs_present[0]))
                                outputs_concat = torch.cat(full_outputs, dim=1)
                                final_output = self.meta_learner(outputs_concat)
                            elif method == "best_single":
                                best_model = min(present_models, key=lambda m: best_val_task_losses[m])
                                j = present_models.index(best_model)
                                final_output = outputs_stack[j]
                            elif method == "greedy_ensemble":
                                chosen = [m for m in getattr(self, "ens_idxs", list(range(self.model_num)))
                                          if m in present_models]
                                if len(chosen) == 0:
                                    final_output = torch.mean(outputs_stack, dim=0)
                                else:
                                    w = weights_all[chosen]
                                    w = w / torch.sum(w)
                                    chosen_pos = [present_models.index(m) for m in chosen]
                                    final_output = torch.sum(
                                        w.to(outputs_stack.device).unsqueeze(1).unsqueeze(2) * outputs_stack[chosen_pos],
                                        dim=0,
                                    )
                            else:
                                raise ValueError(method)

                            prob = torch.softmax(final_output, dim=1).cpu().numpy()
                            pred = torch.argmax(final_output, dim=1).cpu().numpy()
                            records[method]["y_true"].append(target_g.cpu().numpy())
                            records[method]["y_pred"].append(pred)
                            records[method]["y_prob"].append(prob)

        results = {}
        for method, rec in records.items():
            y_true = np.concatenate(rec["y_true"])
            y_pred = np.concatenate(rec["y_pred"])
            y_prob = np.concatenate(rec["y_prob"])
            results[method] = compute_classification_metrics(y_true, y_pred, y_prob, num_classes=self.num_classes)

        results["cohort"] = []
        for rec in cohort_records:
            if len(rec["y_true"]) == 0:
                results["cohort"].append(None)
            else:
                y_true = np.concatenate(rec["y_true"])
                y_pred = np.concatenate(rec["y_pred"])
                y_prob = np.concatenate(rec["y_prob"])
                results["cohort"].append(
                    compute_classification_metrics(y_true, y_pred, y_prob, num_classes=self.num_classes)
                )
        return results


TrainerJointAlzheimer = TrainerJointPolyMNIST


# ============================================================
# BENCHMARKS WITH EXPANDED LATE FUSION
# ============================================================


class BenchmarksWithExpandedLateFusion:
    def __init__(self, config, input_dims, ensemble_methods=None):
        self.config = copy.deepcopy(config)
        self.input_dims = list(input_dims)
        self.model_num = len(input_dims)
        self.num_classes = int(self.config["output_dim"])
        self.use_gpu = bool(self.config["use_gpu"])
        self.device = torch.device("cuda" if self.use_gpu and torch.cuda.is_available() else "cpu")
        self.ensemble_methods = list(ensemble_methods or LATE_FUSION_ENSEMBLE_METHODS)

        self.modality_models = [
            MLP_Net(int(d), HIDDEN_DIMS_PER_MODALITY, self.num_classes).to(self.device)
            for d in self.input_dims
        ]
        self.early_fusion_model = MLP_Net(
            sum(self.input_dims), HIDDEN_DIMS_EARLY_FUSION, self.num_classes
        ).to(self.device)

        self.criterion = nn.CrossEntropyLoss()
        self.best_val_task_losses = [float("inf")] * self.model_num
        self.best_val_early_fusion_loss = float("inf")
        self.ens_idxs = list(range(self.model_num))

    def _train_single_model(self, model, train_loader, val_loader, get_input_fn, label):
        optimizer = optim.Adam(
            model.parameters(),
            lr=self.config["init_lr"],
            weight_decay=self.config["weight_decay"],
        )
        best_state = copy.deepcopy(model.state_dict())
        best_val_loss = float("inf")

        for epoch in range(int(self.config["epochs"])):
            model.train()
            for batch in train_loader:
                modalities, target = batch[:-1], batch[-1]
                modalities = [m.to(self.device) for m in modalities]
                yb = target.to(self.device)
                xb = get_input_fn(modalities)
                optimizer.zero_grad()
                logits = model(xb)
                loss = self.criterion(logits, yb)
                loss.backward()
                optimizer.step()

            model.eval()
            total_loss, total_n = 0.0, 0
            with torch.no_grad():
                for batch in val_loader:
                    modalities, target = batch[:-1], batch[-1]
                    modalities = [m.to(self.device) for m in modalities]
                    yb = target.to(self.device)
                    xb = get_input_fn(modalities)
                    logits = model(xb)
                    loss = self.criterion(logits, yb)
                    bs = yb.size(0)
                    total_loss += float(loss.item()) * bs
                    total_n += bs
            val_loss = total_loss / total_n if total_n > 0 else float("inf")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_state)
        if self.config.get("verbose", False):
            print(f"  [{label}] best val loss = {best_val_loss:.4f}")
        return best_val_loss

    def train(self, train_loader, val_loader):
        print("=" * 80)
        print("Training BenchmarksWithExpandedLateFusion")
        print("=" * 80)
        for i in range(self.model_num):
            print(f"Training per-modality model m{i} ...")
            loss_i = self._train_single_model(
                self.modality_models[i],
                train_loader, val_loader,
                get_input_fn=(lambda mods, idx=i: mods[idx]),
                label=f"modality_{i+1}",
            )
            self.best_val_task_losses[i] = loss_i

        print("Training early-fusion model ...")
        self.best_val_early_fusion_loss = self._train_single_model(
            self.early_fusion_model,
            train_loader, val_loader,
            get_input_fn=(lambda mods: torch.cat(mods, dim=1)),
            label="early_fusion",
        )

        finite_losses = [(i, l) for i, l in enumerate(self.best_val_task_losses) if np.isfinite(l)]
        finite_losses_sorted = sorted(finite_losses, key=lambda x: x[1])
        self.ens_idxs = [i for i, _ in finite_losses_sorted]

    def _get_weights(self, present_models):
        valid = [(m, self.best_val_task_losses[m]) for m in present_models
                 if np.isfinite(self.best_val_task_losses[m])]
        if len(valid) == 0:
            return None
        eps = 1e-8
        inv = np.array([1.0 / (loss + eps) for _, loss in valid], dtype=np.float32)
        inv = inv / inv.sum()
        weight_map = {m: float(w) for (m, _), w in zip(valid, inv)}
        weights = np.array([weight_map.get(m, 0.0) for m in present_models], dtype=np.float32)
        s = weights.sum()
        if s <= 0:
            return None
        weights = weights / s
        return torch.tensor(weights, dtype=torch.float32, device=self.device)

    def _fuse(self, method, outputs_stack, weights):
        if method == "simple_average":
            return torch.mean(outputs_stack, dim=0)
        if method == "weighted_average":
            if weights is None:
                return torch.mean(outputs_stack, dim=0)
            return torch.sum(weights.unsqueeze(1).unsqueeze(2) * outputs_stack, dim=0)
        if method == "majority_voting":
            final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
            return F.one_hot(final_pred, num_classes=self.num_classes).float()
        if method == "weighted_voting":
            if weights is None:
                final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                return F.one_hot(final_pred, num_classes=self.num_classes).float()
            top_preds = torch.argmax(outputs_stack, dim=2)
            num_classes = outputs_stack.shape[2]
            batch_size = outputs_stack.shape[1]
            weighted_votes = torch.zeros((batch_size, num_classes), device=outputs_stack.device)
            for k, model_preds in enumerate(top_preds):
                weighted_votes.scatter_add_(
                    1, model_preds.unsqueeze(1),
                    torch.full((batch_size, 1), float(weights[k]), device=outputs_stack.device),
                )
            return weighted_votes
        if method == "best_single":
            losses = self.best_val_task_losses
            best_model = min(range(self.model_num),
                             key=lambda m: losses[m] if np.isfinite(losses[m]) else float("inf"))
            return outputs_stack[best_model]
        if method == "greedy_ensemble":
            chosen = list(self.ens_idxs)
            if len(chosen) == 0:
                return torch.mean(outputs_stack, dim=0)
            chosen_weights = self._get_weights(chosen)
            if chosen_weights is None:
                return torch.mean(outputs_stack[chosen], dim=0)
            return torch.sum(
                chosen_weights.unsqueeze(1).unsqueeze(2) * outputs_stack[chosen],
                dim=0,
            )
        raise ValueError(method)

    def test(self, test_loader):
        records = {}
        for i in range(self.model_num):
            records[f"modality_{i+1}"] = {"y_true": [], "y_pred": [], "y_prob": []}
        records["early_fusion"] = {"y_true": [], "y_pred": [], "y_prob": []}
        for m in self.ensemble_methods:
            records[f"late_fusion_{m}"] = {"y_true": [], "y_pred": [], "y_prob": []}

        for model in self.modality_models:
            model.eval()
        self.early_fusion_model.eval()

        with torch.no_grad():
            for batch in test_loader:
                modalities, target = batch[:-1], batch[-1]
                modalities = [m.to(self.device) for m in modalities]
                yb = target.to(self.device)

                outputs = []
                for i in range(self.model_num):
                    out = self.modality_models[i](modalities[i])
                    outputs.append(out)
                    prob = torch.softmax(out, dim=1).cpu().numpy()
                    pred = torch.argmax(out, dim=1).cpu().numpy()
                    records[f"modality_{i+1}"]["y_true"].append(yb.cpu().numpy())
                    records[f"modality_{i+1}"]["y_pred"].append(pred)
                    records[f"modality_{i+1}"]["y_prob"].append(prob)

                ef_logits = self.early_fusion_model(torch.cat(modalities, dim=1))
                prob = torch.softmax(ef_logits, dim=1).cpu().numpy()
                pred = torch.argmax(ef_logits, dim=1).cpu().numpy()
                records["early_fusion"]["y_true"].append(yb.cpu().numpy())
                records["early_fusion"]["y_pred"].append(pred)
                records["early_fusion"]["y_prob"].append(prob)

                outputs_stack = torch.stack(outputs)
                weights = self._get_weights(list(range(self.model_num)))
                for method in self.ensemble_methods:
                    final_output = self._fuse(method, outputs_stack, weights)
                    prob = torch.softmax(final_output, dim=1).cpu().numpy()
                    pred = torch.argmax(final_output, dim=1).cpu().numpy()
                    key = f"late_fusion_{method}"
                    records[key]["y_true"].append(yb.cpu().numpy())
                    records[key]["y_pred"].append(pred)
                    records[key]["y_prob"].append(prob)

        results = {}
        for key, rec in records.items():
            y_true = np.concatenate(rec["y_true"])
            y_pred = np.concatenate(rec["y_pred"])
            y_prob = np.concatenate(rec["y_prob"])
            results[key] = compute_classification_metrics(
                y_true, y_pred, y_prob, num_classes=self.num_classes
            )
        return results


# ============================================================
# MODEL BUILDERS
# ============================================================


def build_models(input_dims):
    if len(input_dims) != NUM_MODALITIES:
        raise ValueError(f"Expected {NUM_MODALITIES} input dims, got {len(input_dims)}")
    return [
        MLP_Net(int(d), HIDDEN_DIMS_PER_MODALITY, int(BASE_CONFIG["output_dim"]))
        for d in input_dims
    ]


# ============================================================
# RESULT HELPERS
# ============================================================


def flatten_results(family, training_mode, setting_name, results, repetition, split_seed):
    rows = []
    for k, v in results.items():
        if k == "cohort":
            for i, item in enumerate(v):
                if item is None:
                    continue
                if isinstance(item, dict):
                    rows.append({
                        "repetition": repetition, "split_seed": split_seed,
                        "family": family, "training_mode": training_mode,
                        "setting": setting_name, "method": f"cohort_{i}", **item,
                    })
                else:
                    rows.append({
                        "repetition": repetition, "split_seed": split_seed,
                        "family": family, "training_mode": training_mode,
                        "setting": setting_name, "method": f"cohort_{i}",
                        "value": float(item),
                    })
        else:
            if isinstance(v, dict):
                rows.append({
                    "repetition": repetition, "split_seed": split_seed,
                    "family": family, "training_mode": training_mode,
                    "setting": setting_name, "method": k, **v,
                })
            else:
                rows.append({
                    "repetition": repetition, "split_seed": split_seed,
                    "family": family, "training_mode": training_mode,
                    "setting": setting_name, "method": k, "value": float(v),
                })
    return rows


def apply_requested_setting_order(df: pd.DataFrame) -> pd.DataFrame:
    setting_order = [
        "filtered",
        "imputation_filteredpart",
        "imputation_extrapart",
        "imputation_overall",
    ]
    df = df.copy()
    df["setting"] = pd.Categorical(df["setting"], categories=setting_order, ordered=True)
    sort_cols = [c for c in ["repetition", "split_seed", "setting", "family", "training_mode", "method"]
                 if c in df.columns]
    return df.sort_values(sort_cols).reset_index(drop=True)


def compute_standard_error(series: pd.Series) -> float:
    x = series.dropna().astype(float)
    n = len(x)
    if n <= 1:
        return np.nan
    return float(x.std(ddof=1) / np.sqrt(n))


def average_results_over_repetitions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    group_cols = ["family", "training_mode", "setting", "method"]
    metric_candidates = ["accuracy", "macro_f1", "auc_ovr", "sensitivity", "specificity", "value"]
    metric_cols = [c for c in metric_candidates if c in df.columns]

    agg_dict = {}
    for metric in metric_cols:
        agg_dict[f"{metric}_mean"] = (metric, "mean")
        agg_dict[f"{metric}_se"] = (metric, compute_standard_error)

    avg_df = df.groupby(group_cols, dropna=False, observed=True).agg(**agg_dict).reset_index()
    rep_counts = (
        df.groupby(group_cols, dropna=False, observed=True)["repetition"]
        .nunique().reset_index(name="num_repetitions")
    )
    avg_df = avg_df.merge(rep_counts, on=group_cols, how="left")
    avg_df = apply_requested_setting_order(avg_df)
    return avg_df


# ============================================================
# MAIN EXPERIMENT
# ============================================================


def run_full_experiment_one_seed(repetition: int, split_seed: int):
    print("\n" + "#" * 100)
    print(f"STARTING REPETITION {repetition + 1}/{NUM_REPETITIONS} | split_seed={split_seed}")
    print("#" * 100)

    seed_everything(split_seed)
    config = get_config_for_seed(split_seed)
    all_rows = []
    settings = build_all_settings(split_seed=split_seed)

    filtered_setting = settings["filtered"]

    train_loader, val_loader, test_loader = make_loaders_from_arrays(
        filtered_setting["train_arrays"],
        filtered_setting["val_arrays"],
        filtered_setting["test_arrays"],
        filtered_setting["y_train"],
        filtered_setting["y_val"],
        filtered_setting["y_test"],
        BATCH_SIZE,
    )
    input_dims = [a.shape[1] for a in filtered_setting["train_arrays"]]

    print(f"[matched-data] all families : "
          f"train={len(filtered_setting['y_train'])}, "
          f"val={len(filtered_setting['y_val'])}, "
          f"test={len(filtered_setting['y_test'])}  (filtered, no sentinels)")

    bm = BenchmarksWithExpandedLateFusion(
        config, input_dims, ensemble_methods=LATE_FUSION_ENSEMBLE_METHODS
    )
    bm.train(train_loader, val_loader)
    all_rows.extend(flatten_results(
        "benchmarks", "na", "filtered",
        bm.test(test_loader), repetition, split_seed,
    ))

    joint_models_m = build_models(input_dims)
    joint_m = TrainerJointPolyMNIST(config, joint_models_m, [train_loader, val_loader])
    joint_m.train("marginal", missing_value=None)
    all_rows.extend(flatten_results(
        "joint", "marginal", "filtered",
        joint_m.test(test_loader, missing_value=None),
        repetition, split_seed,
    ))

    joint_models_s = build_models(input_dims)
    joint_s = TrainerJointPolyMNIST(config, joint_models_s, [train_loader, val_loader])
    joint_s.train("shapley", missing_value=None)
    all_rows.extend(flatten_results(
        "joint", "shapley", "filtered",
        joint_s.test(test_loader, missing_value=None),
        repetition, split_seed,
    ))

    rep_df = pd.DataFrame(all_rows)
    rep_df = apply_requested_setting_order(rep_df)
    print("\nFinished repetition:", repetition + 1)
    print(rep_df.head())
    return rep_df


def run_repeated_experiments():
    all_rep_dfs = []
    for rep_idx, split_seed in enumerate(REPETITION_SEEDS):
        rep_df = run_full_experiment_one_seed(repetition=rep_idx, split_seed=split_seed)
        all_rep_dfs.append(rep_df)

    raw_results_df = pd.concat(all_rep_dfs, axis=0, ignore_index=True)
    avg_results_df = average_results_over_repetitions(raw_results_df)
    raw_results_df = apply_requested_setting_order(raw_results_df)
    avg_results_df = apply_requested_setting_order(avg_results_df)
    return raw_results_df, avg_results_df


if __name__ == "__main__":
    seed_everything(RANDOM_STATE)

    print("USE_GPU =", USE_GPU)
    print("MMNIST_ROOT =", MMNIST_ROOT)
    print("NUM_MODALITIES =", NUM_MODALITIES, "NUM_CLASSES =", NUM_CLASSES)
    print("N_SAMPLES_TOTAL =", N_SAMPLES_TOTAL)
    print("APPLY_COMPLEMENTARITY =", APPLY_COMPLEMENTARITY)
    print("COMPLEMENTARITY_REGIONS =", COMPLEMENTARITY_REGIONS)
    print("APPLY_MODALITY_NOISE =", APPLY_MODALITY_NOISE,
          "(types:", MODALITY_NOISE_TYPES, ", levels:", MODALITY_NOISE_LEVELS, ")")
    print("epochs =", BASE_CONFIG["epochs"], "rho_list =", BASE_CONFIG["rho_list"])
    print("NUM_REPETITIONS =", NUM_REPETITIONS)
    print("(matched-data filtered setting only; "
          "all families share the same train/val/test rows)")

    raw_results_df, avg_results_df = run_repeated_experiments()

    raw_out_path = os.path.join(
        DATA_DIR,
        "polymnist_results_filtered_matched_sweep_v1_raw.csv",
    )
    avg_out_path = os.path.join(
        DATA_DIR,
        "polymnist_results_filtered_matched_sweep_v1_avg_with_se.csv",
    )

    raw_results_df.to_csv(raw_out_path, index=False)
    avg_results_df.to_csv(avg_out_path, index=False)

    print("\nRAW RESULTS HEAD:")
    print(raw_results_df.head())
    print("\nAVERAGE RESULTS HEAD:")
    print(avg_results_df.head())
    print(f"\nSaved raw results to: {raw_out_path}")
    print(f"Saved average results to: {avg_out_path}")

TCGA-BRCA preprocessing
  TCGA_DATA_DIR = ./tcga_brca_raw
  OUTPUT_DIR    = ./tcga_brca_processed
  MISSING_VALUE = -999.0
  Top-N features: mRNA=2000, miRNA=None, methyl=5000

Loading mRNA (STAR counts)


FileNotFoundError: [Errno 2] No such file or directory: './tcga_brca_raw/TCGA-BRCA.star_counts.tsv.gz'